# Función de costo en regresión logística

La regresión logística es un modelo de **clasificación binaria**: predice la probabilidad de que una observación pertenezca a la clase positiva. En este notebook derivaremos por qué su función de costo es la *entropía cruzada binaria* (también llamada *log loss*).

## 1. Del puntaje lineal a una probabilidad

Para unas variables de entrada $x$, el modelo calcula primero un puntaje lineal:

$$z = w^T x + b$$

y lo convierte a una probabilidad mediante la sigmoide:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Por tanto, $p = P(y=1 \mid x)$ y queda siempre entre 0 y 1. La etiqueta real $y$ vale 0 o 1.

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
fig = px.line(
    pl.DataFrame({"z": z, "probabilidad": sigmoid(z)}),
    x="z",
    y="probabilidad",
    title="La sigmoide transforma un puntaje en probabilidad",
)
fig.add_hline(y=0.5, line_dash="dash")
fig.show()

## 2. Probabilidad de observar una etiqueta

Si tratamos $y$ como una variable de Bernoulli, la probabilidad de una observación es:

$$P(y \mid x) = p^y(1-p)^{1-y}$$

- Si $y=1$, la expresión queda en $p$: queremos que el modelo asigne alta probabilidad a la clase positiva.
- Si $y=0$, queda en $1-p$: queremos que asigne baja probabilidad a la clase positiva.

Para $n$ observaciones independientes, la verosimilitud es el producto de esas probabilidades. Maximizar productos pequeños es incómodo; tomamos el logaritmo (que conserva el óptimo):

$$\log L = \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)]$$

## 3. La función de costo: log loss

Los optimizadores suelen **minimizar**, así que usamos el negativo del promedio de la log-verosimilitud:

$$J(w,b) = -\frac{1}{n} \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)]$$

Esta es la **entropía cruzada binaria**. Penaliza poco una predicción segura y correcta, pero penaliza muchísimo una predicción segura y equivocada: si $y=1$ y $p$ se acerca a 0, entonces $-\log(p)$ crece sin límite. Es exactamente el comportamiento deseable cuando el modelo afirma una probabilidad imposible para el resultado ocurrido.

In [ ]:
def binary_cross_entropy(y_true, y_prob):
    # Evita log(0) por redondeo de punto flotante.
    eps = np.finfo(float).eps
    p = np.clip(np.asarray(y_prob), eps, 1 - eps)
    y = np.asarray(y_true)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

ejemplos = pl.DataFrame({
    "caso": ["correcta y segura", "correcta e insegura", "incorrecta e insegura", "incorrecta y segura"],
    "y_real": [1, 1, 1, 1],
    "p_clase_positiva": [0.99, 0.60, 0.40, 0.01],
}).with_columns(
    pl.struct(["y_real", "p_clase_positiva"]).map_elements(
        lambda fila: binary_cross_entropy([fila["y_real"]], [fila["p_clase_positiva"]]),
        return_dtype=pl.Float64,
    ).alias("costo")
)
ejemplos

In [ ]:
p = np.linspace(0.001, 0.999, 300)
costos = pl.DataFrame({
    "probabilidad_predicha": np.concatenate([p, p]),
    "costo": np.concatenate([-np.log(p), -np.log(1 - p)]),
    "etiqueta_real": ["y = 1"] * len(p) + ["y = 0"] * len(p),
})
px.line(
    costos,
    x="probabilidad_predicha",
    y="costo",
    color="etiqueta_real",
    title="La log loss castiga las predicciones seguras equivocadas",
).show()

## 4. Gradiente que se optimiza

Para una observación, al derivar la pérdida respecto al puntaje $z$ se obtiene una expresión especialmente simple:

$$\frac{\partial J}{\partial z} = p - y$$

Por la regla de la cadena, para todo el conjunto:

$$\nabla_w J = \frac{1}{n}X^T(p-y), \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i(p_i-y_i)$$

La actualización por descenso de gradiente mueve los parámetros para reducir el error probabilístico. En la práctica, `scikit-learn` resuelve esta optimización internamente.

## 5. Ideas clave

- La regresión logística modela probabilidades, no solo etiquetas.
- Su costo procede de máxima verosimilitud para una variable Bernoulli.
- La entropía cruzada binaria evalúa la calidad de las probabilidades predichas.
- Durante una implementación numérica hay que evitar $\log(0)$; por eso se recortan las probabilidades o se usan implementaciones estables.
- Para evaluar un clasificador completo, complementa la log loss con métricas como precisión, *recall*, F1, ROC-AUC y una matriz de confusión, según el problema.

**Ejercicio:** cambia las cuatro probabilidades de `ejemplos`. ¿Qué pasa con el costo cuando una observación positiva recibe $p=0.001$? Luego crea un ejemplo con $y=0$ y compara.